<a href="https://colab.research.google.com/github/heetaamin/Assignment-3/blob/main/Yeast/modelling_prep.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np

data_dir = '/content/drive/MyDrive/Colab Notebooks/Machine Learning/Assignment 3/Yeast/'
df = pd.read_csv(data_dir + "yeast_clean.csv")
feature_cols = [c for c in df.columns if c != "class"]
df.shape

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


(1453, 9)

In [ ]:
minority_to_majority_order = ['ERL', 'POX', 'VAC', 'EXC', 'ME1', 'ME2', 'ME3', 'MIT', 'NUC', 'CYT']

# Split the data in train, and validation once

In [ ]:
from sklearn.model_selection import train_test_split

X, y = df[feature_cols], df["class"]

X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=0.20, stratify=y_trainval, random_state=42
)

print(f"Train: {len(X_train)}  Val: {len(X_val)}  Test: {len(X_test)}")

Train: 929  Val: 233  Test: 291


# Doing a per class check

In [ ]:
split_summary = pd.DataFrame({
    "train": y_train.value_counts().sort_index(),
    "val": y_val.value_counts().sort_index(),
    "test": y_test.value_counts().sort_index(),
}).fillna(0).astype(int)
split_summary["total"] = split_summary.sum(axis=1)
split_summary = split_summary.loc[minority_to_majority_order]
split_summary

,train,val,test,total
class,,,,
ERL,3,1,1,5
POX,13,3,4,20
VAC,19,5,6,30
EXC,22,6,7,35
ME1,27,7,9,43
ME2,33,8,10,51
ME3,104,26,32,162
MIT,156,39,49,244
NUC,272,68,85,425


# Saving this split

In [ ]:
train_df = X_train.copy(); train_df["class"] = y_train
val_df   = X_val.copy();   val_df["class"]   = y_val
test_df  = X_test.copy();  test_df["class"]  = y_test

train_df.to_csv(data_dir + "split_train.csv", index=False)
val_df.to_csv(data_dir + "split_val.csv", index=False)
test_df.to_csv(data_dir + "split_test.csv", index=False)

print("Saved split_train.csv, split_val.csv, split_test.csv to Drive.")
print("Every later notebook (baseline, incremental) loads these three files directly — never re-splits.")

Saved split_train.csv, split_val.csv, split_test.csv to Drive.
Every later notebook (baseline, incremental) loads these three files directly — never re-splits.


# Standardise the training data only

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

# Encoding the target labels in a minority to majority order

In [ ]:
label_to_index = {label: i for i, label in enumerate(minority_to_majority_order)}
index_to_label = {i: label for label, i in label_to_index.items()}

y_train_idx = y_train.map(label_to_index).values
y_val_idx = y_val.map(label_to_index).values
y_test_idx = y_test.map(label_to_index).values

print(label_to_index)

{'ERL': 0, 'POX': 1, 'VAC': 2, 'EXC': 3, 'ME1': 4, 'ME2': 5, 'ME3': 6, 'MIT': 7, 'NUC': 8, 'CYT': 9}
